# Analyse Span Boundary Error Patterns

Load TSV file with manually tagged errors.

In [78]:
import pandas as pd

error_analysis_file = '../data/error_analysis.tsv'
error_df = pd.read_csv(error_analysis_file, sep='\t')
error_df.shape

(1943, 16)

Remove rows that were not analysed (column `error_type` is empty):

In [79]:
error_df = error_df[error_df.error_type.notna()]
error_df.shape

(312, 16)

In [80]:
error_df['lang'] = error_df.lang.str.upper()

How many tagged errors per language?

In [81]:
error_df.groupby('lang').lang.count()

lang
DE    58
EN    55
FR    67
IT    67
NL    65
Name: lang, dtype: int64

## Touching Spans

The set includes cases where the predicted span doesn't overlap with the test span, but touches it, i.e. the token(s) directly preceding or following the test span.

In [82]:
touch_df = error_df[(error_df.error_type.str.contains('before_')) | (error_df.error_type.str.contains('after_'))]
touch_df.shape

(28, 16)

In [83]:
touch_df.groupby('lang').error_type.value_counts()

lang  error_type    
DE    after_adj         1
      before_adv        1
      before_det        1
EN    before_adj        2
      after_noun        1
      after_pp          1
FR    before_adj        2
      before_pron       2
      after_adj         1
      after_noun        1
      before_adp        1
      before_adv        1
      before_det        1
      before_noun       1
IT    after_adj         2
      before_adj        2
      after_pron        1
      after_verb        1
      before_det        1
      before_noun       1
NL    after_noun        1
      after_verb        1
      before_det_adj    1
Name: count, dtype: int64

So models also regulary predict that a certain entity type is expressed but miss the correct span.

## Partial Overlap
Next step: remove rows with touching spans to focus only on partially overlapping spans.

In [84]:
error_df = error_df[~error_df.index.isin(touch_df.index)]


There are cases where a single entity span in the test set corresponds to multiple spans in the predicted set. For error analysis, we want to calculate errors relative to the total number of spans in the test set. Therefore, we calculate the number of test spans.

In [85]:
cols = [
    'test_start', 'test_end', 'test_text', 'test_label', 
    'pred_start', 'pred_end', 'pred_text', 'pred_label',  
    'match_type',
    'error_type',
    'overlap_start', 'overlap_end', 
    'lang', 'fold', 'text_id', 'sent_idx',
    # 'error_types', 'error_main_types', 'error_pos_types'
]

test_id_cols = [
    'test_start', 'test_end', 'test_text', 'test_label', 'lang', 'fold', 'text_id', 'sent_idx',
]
error_df[test_id_cols].shape, error_df[test_id_cols].drop_duplicates().shape

((284, 8), (238, 8))

How many manually tagged partially overlapping spans?

In [86]:
error_df.lang.value_counts()

lang
NL    62
IT    59
FR    57
DE    55
EN    51
Name: count, dtype: int64

When a test span corresponds to multiple predicted spans or vice versa, there are multiple rows. So to know how many distinct test spans are incorrectly identified, we need to focus on distinct test spans:

In [87]:
test_per_lang = error_df[test_id_cols].drop_duplicates().lang.value_counts()
test_per_lang

lang
IT    51
NL    51
DE    49
FR    44
EN    43
Name: count, dtype: int64

We used the following categories of error types:

- `add_POS`: the predicted span adds a token or a phrase with specific Part-Of-Speech tags with respect to the test span (so NP, PP or VP for noun, prepositional and verb phrases respectively.
- `skip_POS`: the predicted span skips a token or a phrase with specific Part-Of-Speech tags with respect to the test span.
- `before_POS`: the predicted span directly precedes the test span and has a specific POS tag.
- `after_POS`: the predicted span directly follows the test span and has a specific POS tag.
- `one_test_aligns_with_multi_pred`: a single test span corresponds exactly with multiple predicted spans.
- `one_pred_aligns_with_multi_test`: a single predicted span corresponds exactly with multiple test spans.
- `one_test_encompasses_multi_pred`: a single test span contains multiple predicted spans, plus some other token(s).
- `one_pred_encompasses_multi_test`: a single predicted span contains multiple test spans, plus some other token(s).


In [97]:
def get_main_type(error_type):
    if error_type.startswith('add_'):
        return 'Add'
    elif error_type.startswith('skip_'):
        return 'Skip'
    elif error_type.startswith('before_'):
        return 'Before'
    elif error_type.startswith('after_'):
        return 'After'
    elif error_type.endswith('multi_test'):
        #return error_type.replace('one_pred_', '')
        #return 'pred_is_multi_test' if 'aligns_with' in error_type else 'pred_has_multi_test'
        return 'P>GT'
    elif error_type.endswith('multi_pred'):
        #return error_type.replace('one_test_', '')
        #return 'test_is_multi_pred' if 'aligns_with' in error_type else 'test_has_multi_pred'
        return 'GT>P'
    else:
        print(error_type)


def get_pos_type(error_type):
    if error_type.startswith('add_'):
        return error_type.split('_')[-1]
    elif error_type.startswith('skip_'):
        return error_type.split('_')[-1]
    elif error_type.startswith('before_'):
        return error_type.split('_')[-1]
    elif error_type.startswith('after_'):
        return error_type.split('_')[-1]
    else:
        return None


error_df['error_types'] = error_df.error_type.apply(lambda x: x.split('; '))
error_df['error_main_types'] = error_df.error_types.apply(lambda x_list: [get_main_type(x) for x in x_list])
error_df['error_pos_types'] = error_df.error_types.apply(lambda x_list: [get_pos_type(x) for x in x_list])



### Main categories of errors

In [98]:
test_error_df = error_df[test_id_cols + ['error_main_types']].explode('error_main_types').drop_duplicates()

main_count = (test_error_df
    .explode('error_main_types')
    .groupby('lang')
    .error_main_types
    .value_counts()
    .unstack()
    .fillna(0.0))

main_count.columns
main_count = (main_count.T / test_per_lang).T
main_count['# Spans'] = test_per_lang
main_types = [
    '# Spans',
    'Add', 'Skip',
    #'pred_has_multi_test', 'pred_is_multi_test',
    #'test_has_multi_pred', 'test_is_multi_pred'
    'GT>P', 'P>GT'
]
main_count = main_count[main_types]
main_count

error_main_types,# Spans,Add,Skip,GT>P,P>GT
lang,,,,,
DE,49,0.306122,0.510204,0.122449,0.081633
EN,43,0.488372,0.302326,0.186047,0.046512
FR,44,0.000000,0.818182,0.181818,0.000000
IT,51,0.176471,0.686275,0.156863,0.000000
NL,51,0.254902,0.627451,0.156863,0.000000


In [105]:

print(main_count.style.format(precision=2).to_latex())


\begin{tabular}{lrrrrr}
error_main_types & # Spans & Add & Skip & GT>P & P>GT \\
lang &  &  &  &  &  \\
DE & 49 & 0.31 & 0.51 & 0.12 & 0.08 \\
EN & 43 & 0.49 & 0.30 & 0.19 & 0.05 \\
FR & 44 & 0.00 & 0.82 & 0.18 & 0.00 \\
IT & 51 & 0.18 & 0.69 & 0.16 & 0.00 \\
NL & 51 & 0.25 & 0.63 & 0.16 & 0.00 \\
\end{tabular}



### Part-of-speech of added/skipped tokens

To understand if there are specific linguistic patterns associated with parts of speech, we zoom in the POS tag of added/skipped tokens (or phrase types for added/skipped multi-word phrases).

In [49]:
pos_error_df = (error_df
    .explode('error_pos_types')
    .groupby('lang')
    .error_pos_types
    .value_counts()
    .unstack()
    .fillna(0.0))

pos_freq = pos_error_df.sum().sort_values(ascending=False)
pos_freq

error_pos_types
adj             46.0
det             33.0
prep            25.0
np              23.0
noun            17.0
pp              15.0
adv             13.0
vp              10.0
verb             9.0
continuation     5.0
pron             4.0
propn            4.0
adp              2.0
parenthesis      1.0
conj             1.0
punct            1.0
num              1.0
dtype: float64

There's a drop in frequency after verb. Let's focus on the most common POS/phrase types:

In [106]:
pos_order = list(pos_freq[pos_freq >= 9].index)
pos_error_df[pos_order]

error_pos_types,adj,det,prep,np,noun,pp,adv,vp,verb
lang,,,,,,,,,
de,2.0,11.0,7.0,2.0,2.0,1.0,7.0,2.0,2.0
en,5.0,1.0,9.0,5.0,2.0,5.0,2.0,3.0,2.0
fr,13.0,1.0,2.0,7.0,5.0,4.0,0.0,2.0,0.0
it,22.0,7.0,3.0,5.0,4.0,3.0,0.0,0.0,1.0
nl,4.0,13.0,4.0,4.0,4.0,2.0,4.0,3.0,4.0


In [107]:
add_skip_count = (error_df
    .explode('error_types')
    .groupby('lang')
    .error_types
    .value_counts()
    .unstack()
    .fillna(0.0))

pos_cols = [col for col in add_skip_count.columns if col.count('_') == 1 and col.split('_')[-1] in pos_order]
pos_cols = sorted(pos_cols, key=lambda x: pos_order.index(x.split('_')[-1]))

add_skip_count[pos_cols]


error_types,add_adj,skip_adj,add_det,skip_det,add_prep,skip_prep,add_np,skip_np,add_noun,skip_noun,add_pp,skip_pp,add_adv,skip_adv,add_vp,skip_vp,add_verb,skip_verb
lang,,,,,,,,,,,,,,,,,,
DE,0.0,0.0,6.0,4.0,3.0,2.0,0.0,2.0,0.0,0.0,0.0,1.0,3.0,4.0,1.0,1.0,0.0,2.0
EN,2.0,2.0,0.0,1.0,5.0,4.0,1.0,4.0,1.0,0.0,4.0,1.0,2.0,0.0,1.0,2.0,2.0,0.0
FR,0.0,13.0,0.0,1.0,0.0,2.0,0.0,7.0,0.0,5.0,0.0,4.0,0.0,0.0,0.0,2.0,0.0,0.0
IT,5.0,16.0,0.0,7.0,1.0,2.0,0.0,5.0,0.0,4.0,0.0,3.0,0.0,0.0,0.0,0.0,1.0,0.0
NL,0.0,3.0,4.0,9.0,1.0,1.0,0.0,4.0,1.0,3.0,1.0,1.0,1.0,3.0,0.0,3.0,1.0,2.0


In [67]:
(add_skip_count[pos_cols].T / test_per_lang).T.style.format(precision=2).background_gradient(axis=None)

error_types,add_adj,skip_adj,add_det,skip_det,add_prep,skip_prep,add_np,skip_np,add_noun,skip_noun,add_pp,skip_pp,add_adv,skip_adv,add_vp,skip_vp,add_verb,skip_verb
lang,,,,,,,,,,,,,,,,,,
de,0.00,0.00,0.12,0.08,0.06,0.04,0.00,0.04,0.00,0.00,0.00,0.02,0.06,0.08,0.02,0.02,0.00,0.04
en,0.05,0.05,0.00,0.02,0.12,0.09,0.02,0.09,0.02,0.00,0.09,0.02,0.05,0.00,0.02,0.05,0.05,0.00
fr,0.00,0.30,0.00,0.02,0.00,0.05,0.00,0.16,0.00,0.11,0.00,0.09,0.00,0.00,0.00,0.05,0.00,0.00
it,0.10,0.31,0.00,0.14,0.02,0.04,0.00,0.10,0.00,0.08,0.00,0.06,0.00,0.00,0.00,0.00,0.02,0.00
nl,0.00,0.06,0.08,0.18,0.02,0.02,0.00,0.08,0.02,0.06,0.02,0.02,0.02,0.06,0.00,0.06,0.02,0.04


For French and Italian, adjectives are the most commonly skipped (30% and 31% of errors respectively). Other notable error groups are determiners for German, Italian and Dutch (8%, 14% and 18% of errors respectively) and noun phrases for English, French, Italian and Dutch (9%, 16%, 10% and 8%). We note that far fewer mistakes are made with adverbs and verbs  (5% and 4% across all five languages respectively).

In [102]:
pos_freq / test_per_lang.sum()

error_pos_types
adj             0.193277
det             0.138655
prep            0.105042
np              0.096639
noun            0.071429
pp              0.063025
adv             0.054622
vp              0.042017
verb            0.037815
continuation    0.021008
pron            0.016807
propn           0.016807
adp             0.008403
parenthesis     0.004202
conj            0.004202
punct           0.004202
num             0.004202
dtype: float64